# IITM Courses Data Crawler

This notebook crawls all courses from https://study.iitm.ac.in/ds/academics.html# and extracts:

- **Course ID** - Extracted from URL
- **Course Name** - Full course title
- **Course Level** - Foundation/Diploma/Degree (based on course ID last 4 digits < 5000)
- **Course Credits** - Number of credits
- **Pre-requisites & Co-requisites**
- **What you'll learn** - Learning outcomes
- **Weekwise Course Structure**

## HTML Structure To Get Course Links

### **Foundation/Diploma Courses**
- **Tag**: `<tr>`
- **Attribute**: `data-url="course_pages/BSMA1001.html"`
- **Pattern**: `<tr class="table-hover-row" data-url="course_pages/..." style="cursor: pointer">`
- **Examples**: BSMA1001, BSCS1001, BSHS1001 (IDs with last 4 digits < 2000 typically)

### **Degree Level Courses**
- **Tag**: `<a>`
- **Attribute**: `href="course_pages/BSCS3001.html"`
- **Pattern**: `<a href="course_pages/...">`Course Name`</a>`
- **Examples**: BSCS3001, BSCS3002, BSCS4001 (IDs with last 4 digits >= 3000)

---

## What Tags & Parameters to Check:

✅ **Extract from `<tr data-url="...">` tags** - Foundation/Diploma courses
✅ **Extract from `<a href="course_pages/...">` tags** - Degree level courses

Both use the pattern: `course_pages/COURSEID.html`

## Course Page HTML Structure Analysis

Based on the sample course page (BSMA1004 - Statistics for Data Science II), here's what we found:

---

### **DATA LAYOUT:**

```
Course ID: BSMA1004 (line 772)
Course Credits: 4 (line 775)
Course Type: Foundational (line 778)
Pre-requisites: BSMA1002, BSMA1001 (lines 781-787)
Co-requisites: BSMA1003 (lines 790-794)
```

### **KEY FINDINGS:**

1. **Course ID, Name, Type** - Always present
2. **Credits** - Always present
3. **Pre-requisites** - ALWAYS PRESENT (either actual prereqs or links listed)
4. **Co-requisites** - Sometimes present, sometimes not (like in this one, it's present)
5. **What you'll learn** - Present as checkmarks in divs (lines 814-876)
6. **Weekwise Structure** - Table with WEEK 1, WEEK 2, etc. (lines 900-989)

### **HTML TAGS TO USE:**

```
✅ Course ID: <p> containing "Course ID:"
✅ Course Credits: <p> containing "Course Credits:"
✅ Course Type: <p> containing "Course Type:"
✅ Pre-requisites: <p> containing "Pre-requisites:"
✅ Co-requisites: <p> containing "Co-requisites:" (may not exist)
✅ What you'll learn: Look for "What you'll learn" heading + checkmark divs
✅ Weekwise: <table> with <tr> containing "WEEK X"
```

---

Include all fields, but they can be empty if missing. Better to have a consistent data structure.



## Step 1: Import Required Libraries

In [60]:
import requests
from bs4 import BeautifulSoup
import json
import re
from urllib.parse import urljoin
from typing import List, Dict, Optional
import time
import os
import pandas as pd

## Step 2: GET ALL Links from Main Page

In [61]:
BASE_URL = "https://study.iitm.ac.in/ds/academics.html#"

# Fetch the main page
response = requests.get(BASE_URL, timeout=10)
soup = BeautifulSoup(response.content, 'html.parser')

all_links = []

# Extract from <tr data-url="..."> - Foundation/Diploma courses
for tr in soup.find_all('tr', attrs={'data-url': True}):
    data_url = tr.get('data-url')
    full_url = urljoin(BASE_URL, data_url)
    all_links.append(full_url)

# Extract from <a href="course_pages/..."> - Degree courses
for a in soup.find_all('a', href=True):
    href = a.get('href')
    if 'course_pages/' in href:
        full_url = urljoin(BASE_URL, href)
        all_links.append(full_url)

# Remove duplicates
all_links = sorted(list(set(all_links)))

# Filter out PG/MTech courses (those starting with 5 or 6)
filtered_links = []
for url in all_links:
    match = re.search(r'/course_pages/([A-Z]+)(\d)', url)
    if match:
        first_digit = match.group(2)
        if first_digit not in ['5', '6']:  # Skip PG (5) and MTech (6)
            filtered_links.append(url)

all_links = filtered_links

print(f"Total links found (after filtering out PG/MTech): {len(all_links)}")

Total links found (after filtering out PG/MTech): 54


## Step 3: Function to Extract Course Data from a Single Page

In [62]:
def extract_course_data(url):
    """Extract all course data from a single course page"""
    try:
        response = requests.get(url, timeout=10)
        soup = BeautifulSoup(response.content, 'html.parser')

        # Extract Course ID from URL
        match = re.search(r'/course_pages/([A-Z]+\d+)\.html', url)
        course_id = match.group(1) if match else ""

        course_data = {
            'Course ID': course_id,
            'Course Name': '',
            'Course Level': '',
            'Course Credits': '',
            'Course Type': '',
            'Pre-requisites': '',
            'Co-requisites': None,
            'What you\'ll learn': [],
            'Weekwise Structure': [],
            'URL': url
        }

        # Determine Course Level based on first digit of numeric part
        if course_id:
            match = re.search(r'(\d)', course_id)
            if match:
                first_digit = match.group(1)
                if first_digit == '1':
                    course_data['Course Level'] = 'Foundation'
                elif first_digit == '2':
                    course_data['Course Level'] = 'Diploma'
                elif first_digit in ['3', '4']:
                    course_data['Course Level'] = 'Degree'

        # Extract Course Name - look for <p class="h2 font-weight-600 text-dark">
        for p in soup.find_all('p'):
            classes = p.get('class', [])
            if 'h2' in classes and 'font-weight-600' in classes:
                text = p.get_text(strip=True)
                if text and len(text) > 3:
                    course_data['Course Name'] = text
                    break

        # Fallback: if still not found, try h1, h2, h3 tags
        if not course_data['Course Name']:
            for h in soup.find_all(['h1', 'h2', 'h3']):
                text = h.get_text(strip=True)
                if course_id not in text and len(text) > 5:
                    course_data['Course Name'] = text
                    break

        # Extract Course Credits
        for p in soup.find_all('p'):
            text = p.get_text(strip=True)
            if 'Course Credits:' in text:
                match = re.search(r'Course Credits:\s*(\d+)', text)
                if match:
                    course_data['Course Credits'] = match.group(1)

        # Extract Course Type
        for p in soup.find_all('p'):
            text = p.get_text(strip=True)
            if 'Course Type:' in text:
                match = re.search(r'Course Type:\s*(.+?)(?:\n|$)', text)
                if match:
                    course_data['Course Type'] = match.group(1).strip()

        # Extract Pre-requisites
        for p in soup.find_all('p'):
            text = p.get_text(strip=True)
            if 'Pre-requisites:' in text:
                prereq_text = text.split('Pre-requisites:')[1].strip()
                course_data['Pre-requisites'] = prereq_text
                break

        # Extract Co-requisites (optional)
        for p in soup.find_all('p'):
            text = p.get_text(strip=True)
            if 'Co-requisites:' in text:
                coreq_text = text.split('Co-requisites:')[1].strip()
                course_data['Co-requisites'] = coreq_text
                break

        # Extract What you'll learn - find the heading first
        what_learn_items = []
        found_heading = False
        for h in soup.find_all(['p', 'h3']):
            if "What you'll learn" in h.get_text():
                found_heading = True
                # Now get all divs with flex-basis 93% after this heading
                parent = h.find_parent(['div', 'section'])
                if parent:
                    for div in parent.find_all('div'):
                        if 'flex-basis: 93%' in div.get('style', ''):
                            item = div.get_text(strip=True)
                            if item:
                                what_learn_items.append(item)
                break

        # Fallback: if not found, search entire page for divs with that style
        if not what_learn_items:
            for div in soup.find_all('div', style=lambda x: x and 'flex-basis: 93%' in x):
                item = div.get_text(strip=True)
                if item and len(item) > 3:
                    what_learn_items.append(item)

        course_data['What you\'ll learn'] = what_learn_items

        # Extract Weekwise Structure
        for table in soup.find_all('table'):
            for tr in table.find_all('tr'):
                cells = tr.find_all('td')
                if len(cells) >= 2:
                    week_cell = cells[0].get_text(strip=True).upper()
                    topic_cell = cells[1].get_text(strip=True)
                    if 'WEEK' in week_cell:
                        course_data['Weekwise Structure'].append({
                            'week': week_cell,
                            'topics': topic_cell
                        })

        return course_data

    except Exception as e:
        print(f"Error extracting {url}: {e}")
        return None

## Step 4: Test Function with One Course

In [66]:
extract_course_data("https://study.iitm.ac.in/ds/course_pages/BSMA1004.html")


{'Course ID': 'BSMA1004',
 'Course Name': 'Statistics for Data Science II',
 'Course Level': 'Foundation',
 'Course Credits': '4',
 'Course Type': 'Foundational',
 'Pre-requisites': 'BSMA1002 - \xa0Statistics for Data Science IBSMA1001 - \xa0Mathematics for Data Science I',
 'Co-requisites': 'BSMA1003 - \xa0Mathematics for Data Science II',
 "What you'll learn": ['Recalling statistical modeling, description of data.',
  'Applying Probability distributions and related concepts to the data sets',
  'Explaining the concept of estimation of parameters.',
  'Solving the problems related to point and interval estimation.',
  'Explaining the concept of Testing of hypothesis related to mean and variance',
  'Analysing the data using simple regression models and setting up relevant hypothesis tests'],
 'Weekwise Structure': [{'week': 'WEEK 1',
   'topics': 'Multiple random variables - Two random variables, Multiple random variables and distributions'},
  {'week': 'WEEK 2',
   'topics': 'Multipl

## Step 5: Loop Through All Links and Extract Data

In [67]:
all_courses = []

print(f"Starting extraction of {len(all_links)} courses...\n")

for idx, url in enumerate(all_links, 1):
    course = extract_course_data(url)
    if course:
        all_courses.append(course)
        course_name = course['Course Name'][:50] if course['Course Name'] else 'Unknown'
        print(f"[{idx:3d}/{len(all_links)}] ✓ {course['Course ID']:10} | {course_name}")
    else:
        print(f"[{idx:3d}/{len(all_links)}] ✗ Failed")

    time.sleep(0.2)  # Be nice to server

print(f"\n✓ Successfully extracted {len(all_courses)} courses")

Starting extraction of 54 courses...

[  1/54] ✓ BSBT4001   | Algorithmic Thinking in Bioinformatics
[  2/54] ✓ BSBT4002   | Big Data and Biological Networks
[  3/54] ✓ BSCS1001   | Computational Thinking
[  4/54] ✓ BSCS1002   | Programming in Python
[  5/54] ✓ BSCS2001   | Database Management Systems
[  6/54] ✓ BSCS2002   | Programming, Data Structures and Algorithms using 
[  7/54] ✓ BSCS2003   | Modern Application Development I
[  8/54] ✓            | Modern Application Development I - Project
[  9/54] ✓ BSCS2004   | Machine Learning Foundations
[ 10/54] ✓ BSCS2005   | Programming Concepts using Java
[ 11/54] ✓ BSCS2006   | Modern Application Development II
[ 12/54] ✓            | Modern Application Development II - Project
[ 13/54] ✓ BSCS2007   | Machine Learning Techniques
[ 14/54] ✓ BSCS2008   | Machine Learning Practice
[ 15/54] ✓            | Machine Learning Practice - Project
[ 16/54] ✓ BSCS3001   | Software Engineering
[ 17/54] ✓ BSCS3002   | Software Testing
[ 18/54] ✓ BSCS

## Step 6: Save All Courses to JSON

In [68]:
output_file = "courses_data.json"

with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(all_courses, f, indent=2, ensure_ascii=False)

print(f"✓ Data saved to: {output_file}")
print(f"✓ Total courses saved: {len(all_courses)}")



✓ Data saved to: courses_data.json
✓ Total courses saved: 54


In [10]:
# get course info

import json

with open("courses_data.json", 'r', encoding='utf-8') as f:
    courses = json.load(f)

# get course info for course id

for course in courses:
    if course['Course ID'] == 'BSMA1004':
        week_stuct1=course['Weekwise Structure']
    if course['Course ID'] == 'BSMA1002':
        week_stuct2=course['Weekwise Structure']

for week in week_stuct2:
    print(week['week'],week['topics'])

for week in week_stuct1:
    print(week['week'],week['topics'])


WEEK 1 Introduction and type of data, Types of data, Descriptive and Inferential
statistics, Scales of measurement
WEEK 2 Describing categorical data
Frequency distribution of
categorical data, Best practices for
graphing categorical data, Mode and median for
categorical variable
WEEK 3 Describing numerical data
Frequency tables for numerical data, Measures of central tendency - Mean, median and mode, Quartiles and percentiles, Measures of dispersion - Range, variance, standard deviation and IQR, Five number summary
WEEK 4 Association between two variables - 
Association between two categorical variables - Using relative frequencies in contingency tables, Association between two numerical variables - Scatterplot, covariance, Pearson correlation coefficient, Point bi-serial correlation coefficient
WEEK 5 Basic principles of counting and factorial concepts - 
Addition rule of counting, Multiplication rule of
counting, Factorials
WEEK 6 Permutations and combinations
WEEK 7 Probability
Bas